# 2 — Opening a RadDB and filtering

Tutorial 1 wrote an archive. This one reads it back and narrows it down.

The key idea: **`RadDB` is one class with two roles.**

| role | how you get it | what it does |
|---|---|---|
| *archive-bound* | `RadDB(archive_dir=...)` | `archive()`, `open()`, `inventory()`, LUT accessors |
| *data-carrying* | whatever `open()` returns | holds the gates; `filter`, `crop_*`, plots, converters |

Every operation on a data-carrying RadDB returns a **new** RadDB, so calls chain
and nothing is ever mutated underneath you.

---

In [1]:
import os
from pathlib import Path

# --------------------------------------------------------------------------
# CONFIGURATION — point these at your own data
# --------------------------------------------------------------------------
# RadDB is network-agnostic: any xarray DataTree with the standard xradar
# layout works.  These tutorials use two MeteoSwiss volumes and two NEXRAD
# volumes stored as Zarr.  Set the environment variables, or edit the paths.

MCH_DIR    = Path(os.environ.get("RADDB_DATATREE_DIR", "~/data/RADAR/MCH_datatree")).expanduser()
NEXRAD_DIR = Path(os.environ.get("RADDB_NEXRAD_DIR",   "~/data/RADAR/NEXRAD_datatree")).expanduser()

# Where the archive is written.  Anywhere you like — it is just a directory.
ARCHIVE_DIR = Path(os.environ.get("RADDB_TUTORIAL_ARCHIVE",
                                  Path(os.environ.get("TMPDIR", "/tmp")) / "raddb_tutorial_archive"))

print("MCH DataTrees   :", MCH_DIR)
print("NEXRAD DataTrees:", NEXRAD_DIR)
print("Archive         :", ARCHIVE_DIR)


MCH DataTrees   : /data/RADAR/MCH_datatree
NEXRAD DataTrees: /data/RADAR/NEXRAD_datatree
Archive         : /tmp/raddb_tutorial_archive


In [2]:
import warnings
warnings.filterwarnings("ignore")

import polars as pl
import raddb

In [3]:
# This notebook stands on its own: build the archive if tutorial 1 has not run.
if not (ARCHIVE_DIR / "L" / "LUT").exists():
    print("building the archive (see tutorial 1) ...")
    raddb.RadDB(archive_dir=ARCHIVE_DIR, crs=2056).archive(datatree_dir=MCH_DIR)
else:
    print("archive already present:", ARCHIVE_DIR)


archive already present: /tmp/raddb_tutorial_archive


## 1. `open()` — reading the archive

Reading never needs a CRS: it is recovered from the archive itself.

In [4]:
db = raddb.RadDB(archive_dir=ARCHIVE_DIR)      # no crs= needed to read
rdf = db.open(radars="L")
rdf

RadDB [347,449 gates]
  radars     : ['L']
  time range : 2024-08-26 02:45:09+00:00 .. 2024-08-26 02:45:09+00:00
  columns    : gate_id:Int64, time:Datetime(time_unit='ns', time_zone=None), DBZH:Float32, DBZH_raw:Float32, ZDR:Float32, ZDR_raw:Float32, KDP:Float32, RHOHV:Float32, PHIDP:Float32, HC_MCH:Float32, HC_PYART:Float32, HZT:Float32 (+3 more)
  archive_dir: /tmp/raddb_tutorial_archive

`open()` narrows *before* anything is loaded — the time range, the radars and the
columns are all pushed down into the Parquet scan, so you never pay for data you
did not ask for.

In [5]:
# Only two moments, only radar L
small = db.open(radars="L", columns=["DBZH", "ZDR"])
print("columns:", small.columns())

# A time period — any pandas-parseable pair, or a single day
day = db.open(radars="L", time_period=("2024-08-26", "2024-08-27"))
print("gates in the period:", f"{len(day):,}")

columns: ['gate_id', 'DBZH', 'ZDR', 'volume_time', 'radar']
gates in the period: 347,449


In [6]:
# Filters can be pushed down at open() too, so filtered-out rows are never materialised
strong = db.open(radars="L", filters={"var": "DBZH", "logic": ">", "threshold": 30})
print(f"{len(rdf):,} gates -> {len(strong):,} with DBZH > 30 dBZ")

347,449 gates -> 123,008 with DBZH > 30 dBZ


## 2. What you are holding

The data lives in `.data` as a **polars** DataFrame. polars is the backend
throughout RadDB — the read path, the LUT, the archive writer.

In [7]:
print("type:", type(rdf.data).__name__)
print("shape:", rdf.data.shape)
rdf.data.head(3)

type: DataFrame
shape: (347449, 15)


gate_id,time,DBZH,DBZH_raw,ZDR,ZDR_raw,KDP,RHOHV,PHIDP,HC_MCH,HC_PYART,HZT,TEMP,volume_time,radar
i64,datetime[ns],f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,"datetime[μs, UTC]",str
21010005000249,2024-08-26 02:46:08.050,0.5,0.5,NaN,NaN,NaN,NaN,NaN,NaN,4.0,3891.666748,14.733334,2024-08-26 02:45:09 UTC,"""L"""
21010005000749,2024-08-26 02:46:08.050,11.0,11.0,-2.821272,-2.82151,NaN,0.88415,-26.849052,NaN,4.0,3891.666748,14.746333,2024-08-26 02:45:09 UTC,"""L"""
21010005001249,2024-08-26 02:46:08.050,17.0,17.0,1.830259,1.82939,NaN,0.908594,-16.505135,NaN,4.0,3891.666748,14.752833,2024-08-26 02:45:09 UTC,"""L"""


In [8]:
print("radars    :", rdf.radars())
print("variables :", rdf.columns())
print("time range:", rdf.start_time(), "->", rdf.end_time())
print("lon/lat    :", [round(v, 3) for v in rdf.geographic_extent()])
print("archive CRS:", rdf.crs())          # recovered from the archive itself

radars    : ['L']
variables : ['gate_id', 'time', 'DBZH', 'DBZH_raw', 'ZDR', 'ZDR_raw', 'KDP', 'RHOHV', 'PHIDP', 'HC_MCH', 'HC_PYART', 'HZT', 'TEMP', 'volume_time', 'radar']
time range: 2024-08-26 02:45:09+00:00 -> 2024-08-26 02:45:09+00:00


lon/lat    : [7.31, 11.416, 44.408, 47.314]


archive CRS: EPSG:2056


`geographic_extent()` always works. Its projected counterpart `extent()` returns
the bounding box in the archive's own CRS, but needs that CRS stated on the object:

```python
raddb.RadDB(archive_dir=..., crs=2056).open(radars="L").extent()
```

## 3. `filter()` — threshold on values

A filter is a plain dict: `{"var", "logic", "threshold"}`, where `logic` is one of
`==  !=  >  >=  <  <=`. A **list** of dicts is ANDed.

In [9]:
rain = rdf.filter({"var": "DBZH", "logic": ">", "threshold": 20})
print(f"DBZH > 20        : {len(rain):,} gates")

# Several conditions at once — meteorological echo, not clutter
clean = rdf.filter([
    {"var": "DBZH",  "logic": ">",  "threshold": 20},
    {"var": "RHOHV", "logic": ">=", "threshold": 0.9},
    {"var": "ZDR",   "logic": "<",  "threshold": 4},
])
print(f"+ RHOHV >= 0.9, ZDR < 4 : {len(clean):,} gates")

DBZH > 20        : 204,833 gates
+ RHOHV >= 0.9, ZDR < 4 : 196,414 gates


## 4. `sel()` — select by label, xarray-style

Where `filter()` thresholds *values*, `sel()` selects by **coordinate**: a time, a
sweep, a range window, a longitude/latitude box. Scalars match exactly, `slice`
gives a closed interval, and a list matches any of its members.

In [10]:
print("one sweep      :", f"{len(rdf.sel(sweep=1)):,}")
print("sweeps 1,2,3   :", f"{len(rdf.sel(sweep=[1, 2, 3])):,}")
print("range 10-50 km :", f"{len(rdf.sel(range=slice(10_000, 50_000))):,}")
print("a lon/lat box  :", f"{len(rdf.sel(lon=slice(8.6, 9.0), lat=slice(46.0, 46.4))):,}")

one sweep      : 11,629
sweeps 1,2,3   : 42,036


range 10-50 km : 184,299


a lon/lat box  : 192,726


The clever part: `range`, `azimuth`, `elevation_angle`, `latitude`, `longitude`
and `altitude` are **not stored in the Parquet files** — they live once in the LUT.
`sel()` borrows the column it needs, evaluates the selection, and drops it again,
so selecting on geometry costs no storage.

In [11]:
print("stored per gate:", rdf.columns())
print("also selectable :", ["range", "azimuth", "elevation_angle",
                            "latitude", "longitude", "altitude", "sweep"])

narrow = rdf.sel(sweep=1, range=slice(20_000, 60_000))
print(f"\nsweep 1, 20-60 km: {len(narrow):,} gates "
      f"(columns unchanged: {narrow.columns() == rdf.columns()})")

stored per gate: ['gate_id', 'time', 'DBZH', 'DBZH_raw', 'ZDR', 'ZDR_raw', 'KDP', 'RHOHV', 'PHIDP', 'HC_MCH', 'HC_PYART', 'HZT', 'TEMP', 'volume_time', 'radar']
also selectable : ['range', 'azimuth', 'elevation_angle', 'latitude', 'longitude', 'altitude', 'sweep']

sweep 1, 20-60 km: 2,374 gates (columns unchanged: True)


## 5. Chaining, and immutability

Every call returns a new object, so a pipeline reads top to bottom and the original
is untouched.

In [12]:
pipeline = (
    db.open(radars="L")
      .filter({"var": "DBZH", "logic": ">", "threshold": 15})
      .sel(sweep=[1, 2, 3])
      .sel(range=slice(5_000, 80_000))
)
print(f"original : {len(rdf):,} gates")
print(f"pipeline : {len(pipeline):,} gates")
print(f"original still intact: {len(rdf):,}")

original : 347,449 gates
pipeline : 24,453 gates
original still intact: 347,449


## 6. Computed columns

`add_feature()` adds a column derived from the ones you already have and returns a
new RadDB, so it drops straight into a pipeline. The function receives the polars
frame; return a Series, a numpy array, or a polars expression.

In [13]:
derived = (
    rdf.add_feature("ZDR_lin", lambda df: 10 ** (df["ZDR"] / 10))
       .add_feature("DBZH_dev", lambda df: df["DBZH"] - df["DBZH"].mean())
)
derived.data.select(["DBZH", "ZDR", "ZDR_lin", "DBZH_dev"]).head(3)

DBZH,ZDR,ZDR_lin,DBZH_dev
f32,f32,f32,f32
0.5,NaN,NaN,-24.828373
11.0,-2.821272,0.522243,-14.328373
17.0,1.830259,1.524144,-8.328373


In [14]:
# It behaves like any other column from here on — filter it, plot it, export it.
print(f"{len(derived.filter({'var': 'ZDR_lin', 'logic': '>', 'threshold': 2})):,} gates with ZDR_lin > 2")

14,476 gates with ZDR_lin > 2


If you would rather work in plain polars or pandas, nothing stops you — `.data`
is an ordinary polars frame, and `to_pandas()` gives an ordinary pandas one:

```python
import polars as pl
rdf.data.with_columns((pl.col("DBZH") - pl.col("ZDR")).alias("DIFF"))

df = rdf.to_pandas()
df["DIFF"] = df["DBZH"] - df["ZDR"]
```

The RadDB helpers exist so the result stays a RadDB and keeps chaining.

## 7. Getting the data out

Three converters, for three different jobs.

In [15]:
# 1. pandas — with_geometry merges the per-gate coordinates from the LUT
df = rain.to_pandas(with_geometry=True)
print("to_pandas :", df.shape)
print(list(df.columns))

to_pandas : (204833, 19)
['gate_id', 'time', 'DBZH', 'DBZH_raw', 'ZDR', 'ZDR_raw', 'KDP', 'RHOHV', 'PHIDP', 'HC_MCH', 'HC_PYART', 'HZT', 'TEMP', 'volume_time', 'radar', 'latitude', 'longitude', 'altitude', 'sweep']


In [16]:
# with_polar_coords adds range / azimuth / elevation_angle as well.  Off by
# default because they duplicate what the Cartesian columns already say.
print(list(rain.to_pandas(with_polar_coords=True).columns))

['gate_id', 'time', 'DBZH', 'DBZH_raw', 'ZDR', 'ZDR_raw', 'KDP', 'RHOHV', 'PHIDP', 'HC_MCH', 'HC_PYART', 'HZT', 'TEMP', 'volume_time', 'radar', 'latitude', 'longitude', 'altitude', 'sweep', 'range', 'azimuth', 'elevation_angle']


In [17]:
# 2. geopandas — point geometry per gate, ready for spatial joins or QGIS
gdf = rain.to_geopandas()
print("to_geopandas:", gdf.shape, "| CRS:", gdf.crs)
gdf[["gate_id", "DBZH", "geometry"]].head(3)

to_geopandas: (204833, 20) | CRS: EPSG:4326


,gate_id,DBZH,geometry
0,21010005002249,22.0,POINT (8.83347 46.06099)
1,21010005005249,21.0,POINT (8.83381 46.08797)
2,21010005005749,27.0,POINT (8.83387 46.09247)


In [18]:
# 3. back to a DataTree — the full polar structure, for xarray workflows
dt = rain.sel(sweep=1).to_datatree()
print(dt)

<xarray.DataTree>
Group: /
└── Group: /sweep_1
        Dimensions:          (azimuth: 360, range: 492)
        Coordinates: (12/15)
          * azimuth          (azimuth) float64 3kB 0.5 1.5 2.5 3.5 ... 357.5 358.5 359.5
          * range            (range) float32 2kB 250.0 750.0 ... 2.452e+05 2.457e+05
            latitude         (azimuth, range) float64 1MB 46.04 46.05 ... 48.25 48.25
            longitude        (azimuth, range) float64 1MB 8.833 8.833 ... 8.805 8.805
            altitude         (azimuth, range) float64 1MB 1.625e+03 ... 4.356e+03
            x                (azimuth, range) float64 1MB 2.182 6.545 ... -2.144e+03
            ...               ...
            y_2056           (azimuth, range) float64 1MB 1.1e+06 ... 1.345e+06
            site_latitude    float64 8B 46.04
            site_longitude   float64 8B 8.833
            site_altitude    float64 8B 1.626e+03
            sweep_number     int64 8B 1
            elevation_angle  float64 8B -0.19
        Data 

`to_datatree()` reindexes onto the complete azimuth x range grid, so gates you
filtered out come back as NaN. That is what makes it round-trippable, but it also
makes it much heavier than the other two — prefer `to_pandas` / `to_geopandas`
unless you specifically need xarray.

---
## Recap

```python
db  = raddb.RadDB(archive_dir=...)               # reading needs no CRS
rdf = db.open(radars="L", time_period=(...), columns=[...], filters=...)

rdf.filter({"var": "DBZH", "logic": ">", "threshold": 20})   # by value
rdf.sel(sweep=1, range=slice(10_000, 50_000))                # by label
rdf.add_feature("ZDR_lin", lambda df: 10 ** (df["ZDR"] / 10))

rdf.to_pandas(with_geometry=True); rdf.to_geopandas(); rdf.to_datatree()
```

**Next:** [3 — Areas of interest](03_area_of_interest.ipynb)